# Equipo 19 · Endo-STTN guiado por profundidad

## Prueba: el estimador de profundidad retroalimenta el inpainting de Endo-STTN

Este notebook documenta y reproduce la prueba en la que se juntaron dos modelos:

| Pieza | Papel | Se entrena |
|---|---|---|
| **Endo-STTN** | Inpainting temporal de reflejos especulares | **Sí** |
| **AF-SfMLearner** (estimación de profundidad monocular, entrenado en SCARED) | "Juez" de geometría 3D | **No** (congelado) |
| **C3VDv2** | Videos de colonoscopía convertidos al formato de Endo-STTN | — |

**Idea.** Durante el entrenamiento, el frame que reconstruye Endo-STTN pasa por el modelo de
profundidad congelado. Una pérdida extra, `L_depth`, obliga a que su disparidad coincida con la del
frame limpio. El gradiente de esa pérdida **atraviesa** el modelo de profundidad (sin cambiar sus pesos)
y llega a Endo-STTN, que aprende a rellenar sin deformar la geometría del tejido.

> Versión **local** (laptop, `D:\Git\Endo-STTN`). La versión para Google Colab es `Equipo19_EndoSTTN_Profundidad_colab.ipynb`.

### Flujo de una iteración

```
 HACIA ADELANTE (datos)
 frames limpios ─► frame enmascarado ─► [Endo-STTN] ─► comp_img ─► [AF-SfMLearner congelado] ─► disp_pred ─┐
       │                                  se entrena                                                       ├─► L_depth
       └─────────────────────────────────────────────► [AF-SfMLearner, sin gradiente] ─► disp_gt ──────────┘

 GRADIENTE DE VUELTA
 Endo-STTN ◄── comp_img ◄── atraviesa AF-SfMLearner ◄── L_depth
 (optimG.step() solo actualiza Endo-STTN)

 gen_loss = GAN + hole + valid + depth_weight · L_depth
```

## 0. Configuración local

Requisitos (ya preparados en la laptop):

| Pieza | Ruta |
|---|---|
| Endo-STTN con los archivos del Equipo 19 | `D:\\Git\\Endo-STTN` |
| AF-SfMLearner | `D:\\Git\\AF-SfMLearner` |
| Pesos End-to-end | `D:\\Git\\pesos\\Model_trained_end_to_end` |
| Videos C3VD convertidos | `D:\\Git\\Endo-STTN\\datasets\\EndoSTTN_dataset` (con `train.json` y `test.json`) |
| Preentrenado Endo-STTN | `D:\\Git\\Endo-STTN\\release_model\\pretrained_model\\gen_00009.pth` |

Abre el notebook con el mismo entorno (`venv`) que usas para entrenar.

In [ ]:
%matplotlib inline
from pathlib import Path
import os, sys, json, glob, random, shutil, subprocess

# ---------- Ajusta si hace falta ----------
ENDO_STTN   = Path(r"D:\Git\Endo-STTN")
CONFIG      = "configs/EndoSTTN_depth_prueba_afsfm.json"
VIDEO_EVAL  = "c2_descending_t4_v3"    # video DISTINTO al de entrenamiento, para evaluar
RUN_SMOKE   = False                    # True = correr la prueba de arranque (5 iteraciones)
RUN_TRAIN   = False                    # True = correr el entrenamiento corto (≈70 min)
# ------------------------------------------
PRETRAINED_DIR = ENDO_STTN / "release_model" / "pretrained_model"

os.chdir(ENDO_STTN)
if str(ENDO_STTN) not in sys.path:
    sys.path.insert(0, str(ENDO_STTN))
for f in ["core/depth_loss.py", "core/trainer.py", "train.py", CONFIG]:
    assert Path(f).exists(), f"Falta {f}: ¿estás en la carpeta de Endo-STTN con los archivos del Equipo 19?"

import numpy as np
import torch
import matplotlib.pyplot as plt

cfg = json.load(open(CONFIG))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SAVE_DIR = Path(cfg["save_dir"]) / f"sttn_{Path(CONFIG).stem}"
TRAIN_VIDEOS = json.load(open(Path(cfg["data_loader"]["data_root"]) / cfg["data_loader"]["name"] / "train.json"))

print("Entorno          : local |", ENDO_STTN)
print("Dispositivo      :", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
print("Dataset          :", cfg["data_loader"]["name"], "| videos de entrenamiento:", TRAIN_VIDEOS)
print("Modelo de depth  :", cfg["depth"]["model"], "| depth_weight =", cfg["losses"]["depth_weight"])
print("Warm-up          :", cfg["depth"]["warmup_iters"], "iteraciones | max_frames =", cfg["depth"]["max_frames"])
print("Checkpoints en   :", SAVE_DIR)
if DEVICE == "cpu":
    print("\nAVISO: no hay GPU. train.py no entrena sin GPU (cuenta GPUs con torch.cuda.device_count()).",
          "\nLas secciones 1-3 y 7 sí funcionan en CPU.")

## 1. Datos: C3VDv2 en formato Endo-STTN

Cada video de C3VD se convirtió a `JPEGImages/<video>.zip` (frames 288×288) y `Annotations/<video>.zip`
(máscaras de reflejos por umbral de brillo). El dataloader de Endo-STTN:

1. Toma `sample_length` (5) frames del video.
2. **Desplaza la máscara 50 px** y **quita los reflejos reales**. Así la zona que se tapa es tejido limpio
   y se conoce la respuesta correcta (pseudo ground truth).

Por eso podemos comparar la profundidad del frame reconstruido contra la del frame limpio.

In [ ]:
from core.dataset import Dataset
%matplotlib inline
# (core/utils.py de Endo-STTN fuerza matplotlib.use("agg"); esto vuelve a mostrar las figuras)

random.seed(0)
ds_train = Dataset(cfg["data_loader"], split="train")
frames, masks = ds_train.load_item(0)          # (T,3,H,W) en [-1,1], (T,1,H,W)
T = frames.shape[0]
to_img = lambda x: ((x.permute(1, 2, 0).numpy() + 1) / 2).clip(0, 1)

fig, ax = plt.subplots(3, T, figsize=(3 * T, 9))
for t in range(T):
    img, m = to_img(frames[t]), masks[t, 0].numpy()
    ax[0, t].imshow(img); ax[0, t].set_title(f"frame {t}")
    ax[1, t].imshow(m, cmap="gray"); ax[1, t].set_title(f"máscara ({100*m.mean():.1f}%)")
    ov = img.copy(); ov[m > 0] = 0.6 * ov[m > 0] + 0.4 * np.array([1.0, 0.2, 0.1])
    ax[2, t].imshow(ov); ax[2, t].set_title("zona que rellena Endo-STTN")
for a in ax.ravel(): a.axis("off")
plt.suptitle(f"Video de entrenamiento: {ds_train.video_names[0]}"); plt.tight_layout(); plt.show()

## 2. El juez de profundidad: AF-SfMLearner congelado

`DepthLoss` carga el modelo con su propio código (sin normalizar la entrada, como se entrenó),
lo pone en `eval()` y le quita el gradiente a **todos** sus pesos.

In [ ]:
from core.depth_loss import DepthLoss, scale_invariant_l1

depth_fn = DepthLoss.from_config(cfg["depth"], DEVICE)
n_params = sum(p.numel() for p in depth_fn.parameters())
n_train = sum(p.numel() for p in depth_fn.parameters() if p.requires_grad)
print(f"Entrada del modelo: {depth_fn.in_size} | parámetros: {n_params/1e6:.1f} M | entrenables: {n_train}")

with torch.no_grad():
    disp = depth_fn._disp(frames.to(DEVICE)).cpu()

fig, ax = plt.subplots(2, T, figsize=(3 * T, 6))
for t in range(T):
    ax[0, t].imshow(to_img(frames[t])); ax[0, t].set_title(f"frame {t}")
    ax[1, t].imshow(disp[t, 0], cmap="magma"); ax[1, t].set_title("disparidad (claro = cerca)")
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.show()

## 3. El puente del gradiente

Simulamos un **mal relleno** (ruido dentro de la máscara) y calculamos `L_depth`.
Al hacer `backward()` vemos:

- **dónde** llega el gradiente: solo dentro de la máscara, porque fuera `comp_img` es el frame original;
- que el modelo de profundidad **no recibe** gradiente en sus pesos (está congelado), aunque lo deja pasar.

Este gradiente es exactamente lo que, en el entrenamiento, `optimG` usa para corregir a Endo-STTN.

In [ ]:
x = frames.to(DEVICE)
m = masks.to(DEVICE).float()
pred = (x + 0.6 * torch.randn_like(x)).clamp(-1, 1).requires_grad_()   # "relleno" malo
comp = x * (1 - m) + m * pred                                           # igual que trainer.py

loss = depth_fn(comp, x, m)
loss.backward()

g = pred.grad.abs().sum(1).cpu()                        # magnitud del gradiente por píxel
inside = (g * masks[:, 0]).sum() / g.sum().clamp_min(1e-12)
print(f"L_depth = {loss.item():.4f}")
print(f"Gradiente dentro de la máscara: {100*inside:.1f}% (fuera es 0 por construcción)")
print("¿Algún peso del modelo de profundidad recibió gradiente?",
      any(p.grad is not None for p in depth_fn.parameters()))

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(to_img(comp[0].detach().cpu())); ax[0].set_title("comp_img (relleno malo)")
ax[1].imshow(masks[0, 0], cmap="gray"); ax[1].set_title("máscara")
ax[2].imshow(g[0], cmap="inferno"); ax[2].set_title("|∂L_depth/∂pred| → hacia Endo-STTN")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## 4. Prueba de arranque (5 iteraciones)

`train.py -e` fuerza 5 iteraciones para comprobar que todo el flujo corre en la GPU.

**Resultado obtenido en la laptop** (video `c2_descending_t4_v2`, AF-SfMLearner):

| Métrica | Valor |
|---|---|
| Iteraciones | 5 (+1 de cierre) |
| Velocidad | ≈ 2.1 s/iteración |
| `hole` / `valid` | 0.471 / 0.355 |
| `depth` | 0.000 (esperado: está en warm-up hasta la iteración 200) |
| Checkpoint | `gen_00001.pth` guardado |

Si vuelves a correrla, borra después la carpeta de checkpoints para que el entrenamiento corto no la retome.

In [ ]:
import shutil
if RUN_SMOKE and DEVICE == "cpu":
    print("Sin GPU: train.py no puede entrenar aquí.")
elif RUN_SMOKE:
    r = subprocess.run([sys.executable, "train.py", "--model", "sttn", "--config", CONFIG, "-e"],
                       capture_output=True, text=True)
    print((r.stdout + r.stderr)[-2500:])
    shutil.rmtree(SAVE_DIR, ignore_errors=True)
    print("Checkpoints de la prueba borrados:", SAVE_DIR)
else:
    print("RUN_SMOKE = False (ya se corrió; ver tabla de arriba)")

## 5. Entrenamiento corto desde el preentrenado

Parte de `release_model/pretrained_model/gen_00009.pth`. Con ≈2 s/iteración, 2000 iteraciones tardan
≈70 min; conviene lanzarlo desde la terminal:

```powershell
python train.py --model sttn --config configs/EndoSTTN_depth_prueba_afsfm.json -c release_model/pretrained_model/ -cn 9
```

Si la barra de progreso no arranca en 0 (el preentrenado trae su contador de iteraciones), sube
`"iterations"` a *iteración inicial + 2000*.

Para la comparación, entrena también con `configs/EndoSTTN_prueba_sin_depth.json` (igual, pero con
`"depth_weight": 0`). Sus checkpoints quedan en otra carpeta, así que no se mezclan.

In [ ]:
cmd = [sys.executable, "train.py", "--model", "sttn", "--config", CONFIG,
       "-c", str(PRETRAINED_DIR) + "/", "-cn", "9"]
if RUN_TRAIN and DEVICE == "cpu":
    print("Sin GPU: train.py no puede entrenar aquí.")
elif RUN_TRAIN:
    subprocess.run(cmd, check=True)
else:
    print("RUN_TRAIN = False. Comando:\n ", " ".join(cmd))

## 6. Curvas de pérdida

Se leen los registros de TensorBoard que escribe el trainer (`loss/depth_loss` aparece tras el warm-up).

In [ ]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

def read_scalars(logdir):
    out = {}
    for sub in ["gen", "dis"]:
        files = glob.glob(str(Path(logdir) / sub / "events.*"))
        if not files:
            continue
        ea = EventAccumulator(str(Path(logdir) / sub)); ea.Reload()
        for tag in ea.Tags()["scalars"]:
            ev = ea.Scalars(tag)
            out[tag] = ([e.step for e in ev], [e.value for e in ev])
    return out

curves = read_scalars(SAVE_DIR)
if not curves:
    print(f"Aún no hay registros en {SAVE_DIR} (se escriben cada 100 iteraciones).")
else:
    tags = [t for t in ["loss/hole_loss", "loss/valid_loss", "loss/gan_loss", "loss/depth_loss"] if t in curves]
    fig, ax = plt.subplots(1, len(tags), figsize=(4.5 * len(tags), 3.5))
    for a, tag in zip(np.atleast_1d(ax), tags):
        a.plot(*curves[tag]); a.set_title(tag); a.set_xlabel("iteración"); a.grid(alpha=.3)
    plt.tight_layout(); plt.show()

## 7. Evaluación: preentrenado vs. con profundidad vs. sin profundidad

Sobre un video **que no se usó para entrenar** (`VIDEO_EVAL`), se tapa una zona sana con la máscara
desplazada y cada modelo la rellena. Medimos, **solo dentro de la máscara**:

- **PSNR** del relleno contra el frame limpio (mayor = mejor);
- **error de profundidad**: L1 invariante a escala entre la disparidad del frame rellenado y la del frame limpio
  (menor = mejor). Es la misma medida que se entrena, así que el modelo con profundidad *debería* bajarla;
  el PSNR indica si eso cuesta calidad de imagen.

In [ ]:
import importlib, pandas as pd

def load_netG(path):
    path = Path(path)
    if path.is_dir():                                  # carpeta de entrenamiento -> último checkpoint
        latest = path / "latest.ckpt"
        if not latest.exists():
            return None
        path = path / f"gen_{latest.read_text().split()[-1]}.pth"
    if not path.exists():
        return None
    net = importlib.import_module("model.sttn").InpaintGenerator().to(DEVICE)
    net.load_state_dict(torch.load(path, map_location=DEVICE)["netG"]); net.eval()
    return net

MODELS = {
    "Preentrenado (gen_00009)": PRETRAINED_DIR / "gen_00009.pth",
    "Con profundidad":          SAVE_DIR,
    "Sin profundidad":          Path(cfg["save_dir"]) / "sttn_EndoSTTN_prueba_sin_depth",
}

ds_eval = Dataset(cfg["data_loader"], split="test")
assert VIDEO_EVAL in ds_eval.video_names, f"{VIDEO_EVAL} no está en test.json"
random.seed(1)
ef, em = ds_eval.load_item(ds_eval.video_names.index(VIDEO_EVAL))
ef, em = ef.to(DEVICE), em.to(DEVICE).float()

def psnr_in_mask(a, b, m):
    mse = (((a - b) / 2) ** 2 * m).sum() / (m.sum() * 3).clamp_min(1)
    return (10 * torch.log10(1 / mse.clamp_min(1e-10))).item()

rows, comps = [], {}
with torch.no_grad():
    d_gt = depth_fn._disp(ef)
    for name, path in MODELS.items():
        net = load_netG(path)
        if net is None:
            print(f"[omitido] {name}: no se encontró {path}"); continue
        pred = net((ef * (1 - em)).unsqueeze(0), em.unsqueeze(0))
        comp = ef * (1 - em) + em * pred
        d_pred = depth_fn._disp(comp)
        w = torch.nn.functional.interpolate(em, size=d_pred.shape[-2:], mode="nearest")
        rows.append({"modelo": name,
                     "PSNR en máscara (dB) ↑": round(psnr_in_mask(comp, ef, em), 2),
                     "error profundidad en máscara ↓": round(scale_invariant_l1(d_pred, d_gt, w).item(), 4)})
        comps[name] = (comp[0].cpu(), d_pred[0, 0].cpu())
        del net

pd.DataFrame(rows).set_index("modelo") if rows else print("No hay modelos para evaluar todavía.")

In [ ]:
if comps:
    n = len(comps) + 1
    fig, ax = plt.subplots(2, n, figsize=(3.6 * n, 7))
    ax[0, 0].imshow(to_img(ef[0].cpu())); ax[0, 0].set_title("frame limpio (referencia)")
    ax[1, 0].imshow(d_gt[0, 0].cpu(), cmap="magma"); ax[1, 0].set_title("disparidad referencia")
    for j, (name, (c, d)) in enumerate(comps.items(), 1):
        ax[0, j].imshow(to_img(c)); ax[0, j].contour(em[0, 0].cpu(), [0.5], colors="cyan", linewidths=.6)
        ax[0, j].set_title(name)
        ax[1, j].imshow(d, cmap="magma"); ax[1, j].set_title("disparidad")
    for a in ax.ravel(): a.axis("off")
    plt.tight_layout(); plt.show()

## 8. Cómo leer los resultados

- **Con profundidad < Sin profundidad en error de profundidad** y **PSNR parecido**: la pérdida 3D ayuda a
  rellenar respetando la geometría sin empeorar la imagen. Es el resultado buscado.
- **Error de profundidad menor pero PSNR mucho peor**: `depth_weight` es demasiado alto; probar 0.05.
- **Sin diferencia**: con un solo video y 2000 iteraciones la señal puede ser débil; entrenar con más videos
  de C3VD (`train.json` con varios) o más iteraciones.

**Limitaciones de esta prueba.** Un solo video de entrenamiento, pocas iteraciones y un juez de profundidad
entrenado en SCARED (no en C3VD). Sirve para validar el flujo completo, no como resultado final.

**Siguientes pasos.**
1. Entrenar con varios videos de C3VD y evaluar en otros distintos.
2. Usar la profundidad real de C3VD como objetivo en lugar de la del modelo congelado.
3. Probar MonoIIT (pesos "winner" del equipo 52) como juez cuando estén disponibles.